In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

In [ ]:
SEPTAL_MASKS_DIR = "./data/shriya/SHMOLLI-output-unet-septum-update2/SAM_masks"
SEPTAL_T1_CSV    = "./data/shriya/SHMOLLI-output-unet-septum-update2/percentiles_T1.csv"
MYO_T1_CSV       = "./data/shriya/SHMOLLI-output-unet-myocardium-update2/percentiles_T1.csv"
MYO_QC_LIST = "./data/shriya/SHMOLLI-output-unet-myocardium-update2/quality_ID_list.csv.npy"
OUTPUT_DIR       = "./data/shriya/SHMOLLI-output-unet-septum-update2"

## Septum Quality Control

In [ ]:
print("Running septal QC...")
all_mask_files = sorted([f for f in os.listdir(SEPTAL_MASKS_DIR) if f.endswith(".tiff")])

septal_quality_IDs = []
septal_bad_IDs     = []

for fname in tqdm(all_mask_files):
    patient_id = fname[:7]
    img    = np.array(Image.open(os.path.join(SEPTAL_MASKS_DIR, fname)))
    binary = (img > 0).astype(np.uint8)
    
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    meaningful   = [c for c in contours if cv2.contourArea(c) > 50]
    
    if len(meaningful) == 1:
        septal_quality_IDs.append(patient_id)
    else:
        septal_bad_IDs.append(patient_id)

print(f"Septal QC: {len(septal_quality_IDs)} passing, {len(septal_bad_IDs)} failing "
      f"({100*len(septal_quality_IDs)/len(all_mask_files):.1f}%)")

# Save septal QC list
np.save(os.path.join(OUTPUT_DIR, "quality_ID_list.csv"), septal_quality_IDs)

## Compare with Quality Myocardial Distributions

In [ ]:
# ── 2. LOAD MYOCARDIAL QC LIST ────────────────────────────────────────────────
myo_quality_IDs = [str(x)[:7] for x in np.load(MYO_QC_LIST, allow_pickle=True)]
print(f"\nMyocardial QC list: {len(myo_quality_IDs)} IDs")

# ── 3. INTERSECT ──────────────────────────────────────────────────────────────
overlapping_IDs = set(septal_quality_IDs) & set(myo_quality_IDs)
print(f"Overlapping QC-passing IDs: {len(overlapping_IDs)}")

# ── 4. LOAD T1 METRICS AND FILTER TO OVERLAP ──────────────────────────────────
df_sep = pd.read_csv(SEPTAL_T1_CSV)
df_myo = pd.read_csv(MYO_T1_CSV)

In [ ]:
# Normalise patient ID column — trim to 7 chars
for df in [df_sep, df_myo]:
    df["Patient_ID"] = df["Patient_ID"].astype(str).str[:7]

df_sep_qc = df_sep[df_sep["Patient_ID"].isin(overlapping_IDs)].copy()
df_myo_qc = df_myo[df_myo["Patient_ID"].isin(overlapping_IDs)].copy()

print(f"\nSeptal T1 rows after filtering:     {len(df_sep_qc)}")
print(f"Myocardial T1 rows after filtering: {len(df_myo_qc)}")

# ── 5. MERGE INTO ONE COMPARISON TABLE ────────────────────────────────────────
df_merged = df_sep_qc.merge(df_myo_qc, on="Patient_ID", suffixes=("_septal", "_myocardial"))
print(f"Merged rows (one row per subject):  {len(df_merged)}")

df_sep_qc.to_csv(os.path.join(OUTPUT_DIR, "cleaned_T1_percentiles_septal.csv"),    index=False)
df_myo_qc.to_csv(os.path.join(OUTPUT_DIR, "cleaned_T1_percentiles_myocardial.csv"), index=False)
df_merged.to_csv(os.path.join(OUTPUT_DIR, "septal_vs_myocardial_T1_comparison.csv"), index=False)

In [ ]:
df_sep_qc  = df_sep_qc.drop_duplicates(subset="Patient_ID", keep="first")
df_myo_qc  = df_myo_qc.drop_duplicates(subset="Patient_ID", keep="first")

df_merged = df_sep_qc.merge(df_myo_qc, on="Patient_ID", suffixes=("_septal", "_myocardial"))

print(f"Septal rows after dedup:    {len(df_sep_qc)}")
print(f"Myocardial rows after dedup:{len(df_myo_qc)}")
print(f"Merged rows:                {len(df_merged)}")

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as stats

myo_cols = [c for c in df_merged.columns if "_myocardial" in c]

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for i, myo_col in enumerate(myo_cols):
    x = df_merged["Mean_T1_septal"]
    y = df_merged[myo_col]
    
    # Drop NaNs
    mask = x.notna() & y.notna()
    x, y = x[mask], y[mask]
    
    r, p = stats.pearsonr(x, y)
    rho, p_spearman = stats.spearmanr(x, y)
    
    axes[i].scatter(x, y, alpha=0.1, s=5, color='steelblue')
    axes[i].set_xlabel("Mean Septal T1 (ms)", fontsize=9)
    axes[i].set_ylabel(myo_col.replace("_myocardial", ""), fontsize=9)
    axes[i].set_title(f"{myo_col.replace('_myocardial', '')}\nPearson r={r:.3f}, p={p:.2e}\nSpearman ρ={rho:.3f}, p={p_spearman:.2e}", fontsize=8)

# Hide unused axes
for j in range(len(myo_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Mean Septal T1 vs Myocardial T1 Metrics", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Summary table
print(f"{'Myocardial Metric':<30} {'Pearson r':>10} {'Pearson p':>12} {'Spearman ρ':>12} {'Spearman p':>12}")
print("-" * 80)
for myo_col in myo_cols:
    x = df_merged["Mean_T1_septal"]
    y = df_merged[myo_col]
    mask = x.notna() & y.notna()
    r, p   = stats.pearsonr(x[mask], y[mask])
    rho, p_spearman = stats.spearmanr(x[mask], y[mask])
    print(f"{myo_col.replace('_myocardial', ''):<30} {r:>10.3f} {p:>12.2e} {rho:>12.3f} {p_spearman:>12.2e}")

In [ ]:
import seaborn as sns

# Build correlation matrix: Mean_T1_septal vs all myocardial metrics
compare_cols = ["Mean_T1_septal"] + myo_cols
df_corr = df_merged[compare_cols].rename(columns=lambda c: c.replace("_myocardial", "_myo"))

corr_matrix = df_corr.corr(method="pearson")

# Just the row we care about: Mean_T1_septal vs all myo metrics
septal_corr = corr_matrix["Mean_T1_septal"].drop("Mean_T1_septal").sort_values(ascending=False)

print("Pearson correlation: Mean Septal T1 vs Myocardial metrics")
print(septal_corr.to_string())

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
septal_corr.plot(kind="barh", ax=ax, color="steelblue", edgecolor="black")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pearson r")
ax.set_title("Mean Septal T1 vs Myocardial T1 Metrics")
plt.tight_layout()
plt.show()